In [ ]:
import json
import pandas as pd

def filter_by_frame_mapping(
    df: pd.DataFrame,
    frame_mapping_file: str,
    frame_col: str = "frame",
    mapping_key: str = "mapped_sixcam_frame_indices",
    shift: int = 0,
    keep_cols=None,
):
    """
    Filter a raw DataFrame by frames listed in a JSON mapping.

    Parameters
    ----------
    df : DataFrame
        Raw data containing a frame column (default 'frame').
    frame_mapping_file : str
        Path to JSON file that contains a list under `mapping_key`.
    frame_col : str, default 'frame'
        Column in `df` to match against mapped frames.
    mapping_key : str, default 'mapped_sixcam_frame_indices'
        Key in the JSON whose value is a list of frame indices.
    shift : int, default 0
        Optional shift to add to the mapped indices before filtering
        (use only if your df's frame numbering is offset).
    keep_cols : list[str] or None
        If provided, select only these columns from the filtered rows.

    Returns
    -------
    filtered : DataFrame
        Rows where df[frame_col] ∈ mapped indices (after optional shift).
        Also attaches attrs: {'time_offset': <value or None>}.
    """
    with open(frame_mapping_file, "r") as f:
        m = json.load(f)

    mapped_seq = m.get(mapping_key, [])
    # allow optional integer shift without loops
    if shift != 0:
        mapped_seq = [int(x) + shift for x in mapped_seq]
    mapped_set = set(mapped_seq)

    if frame_col not in df.columns:
        raise ValueError(f"Column '{frame_col}' not found in DataFrame.")

    filtered = df[df[frame_col].isin(mapped_set)].copy()
    if keep_cols is not None:
        missing = set(keep_cols) - set(filtered.columns)
        if missing:
            raise ValueError(f"Missing columns requested in keep_cols: {sorted(missing)}")
        filtered = filtered.loc[:, keep_cols]

    # propagate useful metadata from the JSON
    filtered.attrs["time_offset"] = m.get("time_offset", None)
    return filtered

raw_df = 
mapping_path = ""

filtered_df = filter_by_frame_mapping(raw_df, mapping_path)